In [ ]:
# @title ▶️ CLICK PLAY TO START

import os, sys, subprocess, time, re, glob
import torch
from IPython.display import display, HTML, clear_output

if not torch.cuda.is_available():
    clear_output()
    print("❌ GPU required. Click Runtime -> Change runtime type -> T4 GPU -> Save, then click Play again.")
    raise SystemExit(1)

print("⏳ Starting Studio... (~1 min)")

# ── 1. ComfyUI & Deps ─────────────────────────────────────
if not os.path.exists('/content/ComfyUI'):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", "/content/ComfyUI"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/requirements.txt"], check=True)

if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    subprocess.run(["git", "clone", "https://github.com/city96/ComfyUI-GGUF.git", "/content/ComfyUI/custom_nodes/ComfyUI-GGUF"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/diffusers"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rembg[gpu]", "onnxruntime-gpu", "aria2", "gradio"], check=True)

# ── 2. Pre-download fast default model (Z-Image Turbo) ─────
DIFF   = '/content/ComfyUI/models/diffusion_models'
CLIP   = '/content/ComfyUI/models/clip'
TXTENC = '/content/ComfyUI/models/text_encoders'
VAE    = '/content/ComfyUI/models/vae'
for d in [DIFF, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, name):
    path = os.path.join(dest_dir, name)
    if not (os.path.exists(path) and os.path.getsize(path) > 1024):
        if os.path.exists(path):
            os.remove(path)
        subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url, "-d", dest_dir, "-o", name], check=True)

dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', DIFF, 'z-image-turbo-fp8-e4m3fn.safetensors')
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors', CLIP, 'qwen_3_4b.safetensors')
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', VAE, 'ae.safetensors')

for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)
for f in glob.glob(f'{TXTENC}/*'):
    link = os.path.join(CLIP, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)

# ── 3. Pull FreeFakeStudio & Start ────────────────────────
REPO = 'https://github.com/MuntasirMalek/FreeFakeStudio.git'
if os.path.exists('/content/FreeFakeStudio'):
    os.chdir('/content/FreeFakeStudio')
    subprocess.run(["git", "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO, "/content/FreeFakeStudio"], check=True)
    os.chdir('/content/FreeFakeStudio')

proc = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

gradio_url = None
url_pattern = re.compile(r'(https://[a-zA-Z0-9-]+\.gradio\.live)')

for line in proc.stdout:
    match = url_pattern.search(line)
    if match:
        gradio_url = match.group(1)
        break

if gradio_url:
    web_app_url = f"https://muntasirmalek.github.io/FreeFakeStudio/?backend={gradio_url}"
    clear_output()
    display(HTML(f'''
    <div style="text-align: center; padding: 40px 10px;">
        <a href="{web_app_url}" target="_blank" style="display: inline-block; background: #2563eb; color: #ffffff; padding: 22px 60px; border-radius: 18px; font-size: 28px; font-weight: 900; text-decoration: none; box-shadow: 0 12px 35px rgba(37,99,235,0.45);">
            🚀 OPEN STUDIO 🚀
        </a>
    </div>
    '''))

try:
    for line in proc.stdout:
        pass
except KeyboardInterrupt:
    proc.terminate()
